# __Authentication and Permissions in DRF.__

## This concept page will provide an in-depth understanding of authentication and permissions in Django REST Framework (DRF). It will explore different authentication schemes, such as token-based, session-based, and OAuth, and learn how to implement them in their DRF-powered APIs.

## __Overview__

### __Authentication__ and __permissions__ are critical aspects of API security, ensuring that only authorized users can access and interact with your API resources.DRF provides a robust framework for implementing various authentication schemes and permission policies, allowing you to tailor access control to your specific needs. This concept explores different authentication methods and permission strategies available in DRF, empowering you to build secure and reliable APIs.

## __Topics__

- #### Authentication in DRF
- #### Permission Policies in DRF
- #### Securing API Endpoints with Authentication and Permissions

## __Objectives__

- ### Understand the role and importance of authentication and permissions in API security.
- ### Learn how to implement various authentication methods, including TokenAuthentication, SessionAuthentication, and JWT Authentication.
- ### Explore different permission policies offered by DRF, such as IsAuthenticated, IsAdminUser, and DjangoModelPermissions.
- ### Create custom permission classes to implement granular access control based on specific requirements.

## __Authentication in Django REST Framework__

### Authentication verifies the identity of a user or client attempting to access your API. DRF supports several authentication methods out-of-the-box and allows for custom implementations.

### __DRF provides several authentication schemes to secure your API endpoints, including:__

#### __1. Token Authentication:__ Clients authenticate by providing a unique token in the request headers.
#### __2. Session Authentication:__ Clients authenticate using Django’s built-in session-based authentication.
#### __3. OAuth Authentication:__ Clients authenticate using the OAuth 2.0 protocol, which allows third-party applications to access user data without requiring their credentials.

### You can configure authentication globally in your  <font color="red">settings.py</font> or at the view or viewset level using the <font color="red">authentication_classes</font> attribute.

## __An example of how to implement token-based authentication in DRF:__

In [ ]:
"""
from rest_framework.authentication import TokenAuthentication
from rest_framework.permissions import IsAuthenticated
from rest_framework.views import APIView

class MyAPIView(APIView):
    authentication_classes = [TokenAuthentication]
    permission_classes = [IsAuthenticated]

    def get(self, request):
        # Only authenticated users can access this view
        return Response({'message': 'Hello, authenticated user!'})
"""

#### In this example, the <font color="red">MyAPIView</font> class requires token-based authentication and the <font color="red">IsAuthenticated</font> permission to access the get method.

## __Permission Policies in DRF.__

### DRF provides a wide range of built-in permission classes to control access to your API endpoints, such as:

- #### <font color="red">AllowAny</font>: Allows access to anyone, regardless of authentication status.
- #### <font color="red">IsAuthenticated</font>: Allows access only to authenticated users.
- #### <font color="red">IsAdminUser</font>: Allows access only to users with the is_staff flag set to True.
- #### <font color="red">IsOwner</font>: Allows access only to the owner of the resource.

### You can also create custom permission classes to implement more complex access control logic.An example of a custom permission class:

In [ ]:
"""
from rest_framework.permissions import BasePermission

class IsAdminOrReadOnly(BasePermission):
    def has_permission(self, request, view):
        if request.method in ['GET', 'HEAD', 'OPTIONS']:
            return True
        return request.user.is_staff
"""

### In this example, the <font color="red">IsAdminOrReadOnly</font> permission class allows read-only access to everyone, but requires the user to be an admin (staff user) for any write operations.

## __Securing API Endpoints with Authentication and Permissions.__

### By combining authentication and permissions, you can secure your API endpoints and control access based on user roles and permissions. An example of how to do this:

In [ ]:
"""
from rest_framework.authentication import TokenAuthentication
from rest_framework.permissions import IsAuthenticated, IsAdminUser
from rest_framework.views import APIView

class MyModelListView(APIView):
    authentication_classes = [TokenAuthentication]
    permission_classes = [IsAuthenticated]

    def get(self, request):
        # Only authenticated users can view the list of models
        queryset = MyModel.objects.all()
        serializer = MyModelSerializer(queryset, many=True)
        return Response(serializer.data)

class MyModelCreateView(APIView):
    authentication_classes = [TokenAuthentication]
    permission_classes = [IsAdminUser]

    def post(self, request):
        # Only admin users can create new model instances
        serializer = MyModelSerializer(data=request.data)
        serializer.is_valid(raise_exception=True)
        serializer.save()
        return Response(serializer.data, status=status.HTTP_201_CREATED)
"""

In this example, the <font color="red">MyModelListView</font> requires token-based authentication and the <font color="red">IsAuthenticated</font> permission, which means only authenticated users can view the list of models. The <font color="red">MyModelCreateView</font>, on the other hand, requires token-based authentication and the <font color="red">IsAdminUser</font> permission, which means only admin users can create new model instances.

## __A Complete Example__

### The following example demonstrates the use of authentication and permissions in a Django REST Framework (DRF) application that provides a simple blog post API. The API allows users to list, create, retrieve, update, and delete blog posts. However, it enforces certain access control rules to ensure that only authenticated users can perform these operations, and that users can only modify posts they have created.

### __The key components of this example include:__

#### __A Post model to represent blog posts:__

- A <font color="red">PostSerializer</font>  to handle the serialization and deserialization of Post instances
- A custom <font color="red">IsAuthorOrReadOnly</font>  permission class to control access to Post instances
- Two DRF views (<font color="red">PostListCreateAPIView</font>  and <font color="red">PostRetrieveUpdateDestroyAPIView) </font> that leverage the authentication and permission classes to secure the API endpoints

## <font color="red">models.py</font>

In [ ]:
"""
from django.db import models
from django.contrib.auth.models import User

class Post(models.Model):
    title = models.CharField(max_length=100)
    content = models.TextField()
    author = models.ForeignKey(User, on_delete=models.CASCADE)
    created_at = models.DateTimeField(auto_now_add=True)
"""

## <font color="red">serializers.py</font>

In [ ]:
"""
from rest_framework import serializers
from .models import Post

class PostSerializer(serializers.ModelSerializer):
    class Meta:
        model = Post
        fields = ['id', 'title', 'content', 'author', 'created_at']
"""

## <font color="red">permissions.py</font>

In [ ]:
"""
from rest_framework.permissions import BasePermission

class IsAuthorOrReadOnly(BasePermission):
    def has_object_permission(self, request, view, obj):
        if request.method in ['GET', 'HEAD', 'OPTIONS']:
            return True
        return obj.author == request.user
"""

## <font color="red">views.py</font>

In [ ]:
"""
from rest_framework import generics
from rest_framework.authentication import TokenAuthentication
from rest_framework.permissions import IsAuthenticated
from .models import Post
from .serializers import PostSerializer
from .permissions import IsAuthorOrReadOnly

class PostListCreateAPIView(generics.ListCreateAPIView):
    authentication_classes = [TokenAuthentication]
    permission_classes = [IsAuthenticated, IsAuthorOrReadOnly]
    queryset = Post.objects.all()
    serializer_class = PostSerializer

    def perform_create(self, serializer):
        serializer.save(author=self.request.user)

class PostRetrieveUpdateDestroyAPIView(generics.RetrieveUpdateDestroyAPIView):
    authentication_classes = [TokenAuthentication]
    permission_classes = [IsAuthenticated, IsAuthorOrReadOnly]
    queryset = Post.objects.all()
    serializer_class = PostSerializer
"""

## <font color="red">urls.py</font>

In [ ]:
"""
from django.urls import path
from .views import PostListCreateAPIView, PostRetrieveUpdateDestroyAPIView

urlpatterns = [
    path('posts/', PostListCreateAPIView.as_view(), name='post-list-create'),
    path('posts/<int:pk>/', PostRetrieveUpdateDestroyAPIView.as_view(), name='post-retrieve-update-destroy'),
]
"""

### In this example, we have a Post model that represents a blog post, with a title, content, author, and created_at fields.

- ### The <font color="red"> PostSerializer</font> is responsible for serializing and deserializing the Post model instances.
- ### The <font color="red"> IsAuthorOrReadOnly</font> permission class is a custom permission that allows read-only access to anyone, but only allows the author of the post to perform CRUD operations on it.
- ### The <font color="red"> PostListCreateAPIView</font> handles the list and create operations for the Post model. It requires token-based authentication (<font color="red">TokenAuthentication</font>) and the <font color="red">IsAuthenticated</font> and <font color="red">IsAuthorOrReadOnly</font> permissions. When creating a new post, the perform_create method is overridden to associate the current user as the author of the post.
- ### The <font color="red"> PostRetrieveUpdateDestroyAPIView</font> handles the retrieve, update, and destroy operations for individual Post instances. It also requires token-based <font color="red">authentication</font> and the IsAuthenticated and <font color="red">IsAuthorOrReadOnly</font> permissions.
- ### In the <font color="red">urls.py</font> file, we define the URL patterns for the two views, allowing clients to access the post list and individual post details.

### With this setup, only authenticated users can access the API, and the <font color="red">IsAuthorOrReadOnly</font> permission ensures that users can only perform CRUD operations on posts they have authored. This provides a basic level of security and access control for the API.

##<font color="red">References</font>

[DRF Authentication Documentation](https://www.django-rest-framework.org/api-guide/authentication/)

[DRF Permissions Documentation](https://www.django-rest-framework.org/api-guide/permissions/)

[Tutorial 4: Authentication & Permissions](https://www.django-rest-framework.org/tutorial/4-authentication-and-permissions/)